# 👑 GeoMAS — Research Master Dashboard

Notebook strutturato per la validazione comportamentale del framework GeoMAS.
Organizzato in 7 capitoli allineati alle 4 Research Questions della tesi.

| # | Capitolo | Focus |
|---|---------|-------|
| 0 | Setup & Data Loading | Configurazione e selezione simulazioni |
| 1 | Baseline Telemetry | Panoramica single-run |
| 2 | Scenario Impact (RQ1) | Baseline vs Scenario |
| 3 | Counterfactual Analysis (RQ2) | Fork comparison + prompt audit |
| 4 | Deception & Moral Washing (RQ3) | GovernmentType/Strategy breakdown |
| 5 | Emergent Equilibria (RQ4) | Network evolution & polarization |
| 6 | Technical Monitoring | Token usage & costs |
| 7 | Action Governance | Engine outcomes, presidential approve/veto |

In [ ]:
import duckdb
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import networkx as nx
import numpy as np
import json as json_lib
from collections import Counter

# Style
sns.set_theme(style='darkgrid', context='talk')
plt.rcParams['figure.figsize'] = (16, 8)
COLORS = {'defense': '#e74c3c', 'foreign': '#3498db', 'economy': '#f1c40f', 'satisfaction': '#2ecc71'}

# Database paths (adjust if needed)
METRICS_DB = 'simulation_metrics.duckdb'
MAIN_DB = 'simulation.duckdb'

def query_metrics(query, params=None):
    if params:
        params = [p.item() if hasattr(p, 'item') else p for p in params]
    with duckdb.connect(METRICS_DB, read_only=True) as conn:
        return conn.execute(query, params or []).df()

def query_main(query, params=None):
    if params:
        params = [p.item() if hasattr(p, 'item') else p for p in params]
    with duckdb.connect(MAIN_DB, read_only=True) as conn:
        return conn.execute(query, params or []).df()

print('Libraries loaded. Databases:', METRICS_DB, MAIN_DB)

## ⚙️ Simulation Selector

In [ ]:
# List all available simulations
sims = query_main("SELECT id, name, genesis_seed, simulation_seed, n_nations, total_turns, scenario_json FROM simulation ORDER BY id")
display(sims)

# Configure these IDs
TARGET_ID = sims['id'].iloc[-1] if not sims.empty else None    # Latest run
BASELINE_ID = sims['id'].iloc[-2] if len(sims) > 1 else None   # Previous run (for RQ1/RQ2)

print(f'\nTarget (current analysis): Sim {TARGET_ID}')
print(f'Baseline (reference):      Sim {BASELINE_ID}')

In [ ]:
def load_metrics(sim_id):
    """Load all metrics tables for a simulation."""
    if not sim_id: return None, None, None
    g = query_metrics('SELECT * FROM metrics_global WHERE simulation_id = ? ORDER BY turn', [str(sim_id)])
    n = query_metrics('SELECT * FROM metrics_nation WHERE simulation_id = ? ORDER BY turn, nation_id', [str(sim_id)])
    t = query_metrics('SELECT * FROM metrics_trust WHERE simulation_id = ? ORDER BY turn', [str(sim_id)])
    return g, n, t

def load_behaviors(sim_id):
    """Load behaviors with strategy/government from main DB."""
    return query_main(
        'SELECT * FROM behaviors WHERE simulation_id = ? ORDER BY turn, nation_id', [sim_id]
    )

# Load target data
global_df, nation_df, trust_df = load_metrics(TARGET_ID)
behaviors_df = load_behaviors(TARGET_ID)
print(f'Loaded: {len(global_df)} global rows, {len(nation_df)} nation rows, {len(trust_df)} trust rows, {len(behaviors_df)} behavior rows')

---
# 📊 Chapter 1: Baseline Telemetry
Single-run overview of the target simulation.

### 1.1 Global Deception & Coherence Trends

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))
ax.plot(global_df['turn'], global_df['global_deception_avg'], label='Deception (avg)', color='red', linewidth=2)
ax.plot(global_df['turn'], global_df['global_coherence_avg'], label='Coherence (avg)', color='green', linewidth=2)
ax.fill_between(global_df['turn'], global_df['global_deception_avg'], alpha=0.1, color='red')
ax.fill_between(global_df['turn'], global_df['global_coherence_avg'], alpha=0.1, color='green')
ax.set_ylim(-0.05, 1.05)
ax.set_xlabel('Turn'); ax.set_ylabel('Score')
ax.set_title('Global Agent Quality: Deception vs Coherence')
ax.legend(); plt.show()

### 1.2 Guns vs Butter

In [ ]:
fig, ax1 = plt.subplots(figsize=(16, 6))
ax2 = ax1.twinx()
ax1.plot(global_df['turn'], global_df['global_trade_volume'], color='blue', linewidth=2, label='Trade Volume')
ax2.plot(global_df['turn'], global_df['units_created'], color='orange', linewidth=2, label='Units Created')
ax1.set_xlabel('Turn'); ax1.set_ylabel('Trade Volume', color='blue')
ax2.set_ylabel('Units Created', color='orange')
ax1.set_title('Guns vs Butter: Global Resource Allocation')
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
plt.show()

### 1.3 Power Projection per Nation

In [ ]:
fig, ax = plt.subplots(figsize=(16, 7))
sns.lineplot(data=nation_df, x='turn', y='power_projection', hue='nation_id', ax=ax, linewidth=2)
ax.set_title('Power Projection Evolution'); plt.show()

### 1.4 Resource Trends

In [ ]:
res_list = ['budget', 'food', 'energy', 'materials']
fig, axes = plt.subplots(2, 2, figsize=(18, 12))
for i, res in enumerate(res_list):
    ax = axes[i // 2][i % 2]
    sns.lineplot(data=nation_df, x='turn', y=res, hue='nation_id', ax=ax, legend=(i == 0))
    ax.set_title(f'{res.capitalize()} Stocks')
plt.tight_layout(); plt.show()

### 1.5 Global KPIs Summary

In [ ]:
kpi = global_df.agg({
    'territories_changed_hands': 'sum',
    'units_created': 'sum',
    'units_destroyed': 'sum',
    'global_trade_volume': 'sum',
    'global_deception_avg': 'mean',
    'global_coherence_avg': 'mean',
    'global_satisfaction_avg': 'mean'
}).to_frame('Value').rename(index={
    'territories_changed_hands': 'Total Territory Changes',
    'units_created': 'Total Units Created',
    'units_destroyed': 'Total Units Destroyed',
    'global_trade_volume': 'Total Trade Volume',
    'global_deception_avg': 'Avg Deception',
    'global_coherence_avg': 'Avg Coherence',
    'global_satisfaction_avg': 'Avg Satisfaction'
})

# Violence rate
total_created = global_df['units_created'].sum()
total_destroyed = global_df['units_destroyed'].sum()
kpi.loc['Violence Rate (destroyed/created)'] = total_destroyed / total_created if total_created > 0 else 0

# Civil unrest count
if 'in_civil_unrest' in nation_df.columns:
    kpi.loc['Total Civil Unrest Turn-Nations'] = nation_df['in_civil_unrest'].sum()

display(kpi.style.format('{:.2f}'))

---
# ⚖️ Chapter 2 (RQ1): Scenario Impact Analysis
**RQ1**: *What is the impact of exogenous scenario perturbations on the behavioral and economic trajectory?*

Compares `BASELINE_ID` vs `TARGET_ID` across all key metrics.

In [ ]:
if BASELINE_ID:
    g_base, n_base, t_base = load_metrics(BASELINE_ID)
    
    # Tag each dataframe
    g_target = global_df.assign(sim='Target (Scenario)')
    g_baseline = g_base.assign(sim='Baseline')
    comp_global = pd.concat([g_target, g_baseline])
    
    n_target = nation_df.assign(sim='Target')
    n_baseline = n_base.assign(sim='Baseline')
    comp_nation = pd.concat([n_target, n_baseline])
else:
    print('⚠️ No BASELINE_ID set. Set it above to enable comparison.')
    comp_global = None
    comp_nation = None

### 2.1 Multi-Metric Overlay

In [ ]:
if comp_global is not None:
    fig, axes = plt.subplots(2, 2, figsize=(18, 12))
    metrics = [
        ('global_deception_avg', 'Deception (Global Avg)'),
        ('global_coherence_avg', 'Coherence (Global Avg)'),
        ('global_satisfaction_avg', 'Satisfaction (Global Avg)'),
        ('global_trade_volume', 'Trade Volume'),
    ]
    for i, (col, title) in enumerate(metrics):
        ax = axes[i // 2][i % 2]
        sns.lineplot(data=comp_global, x='turn', y=col, hue='sim', ax=ax, linewidth=2)
        ax.set_title(title)
    plt.suptitle('RQ1: Baseline vs Scenario — Key Metrics', fontsize=16, y=1.01)
    plt.tight_layout(); plt.show()

### 2.2 Trust Heatmap Comparison

In [ ]:
def plot_trust_heatmap(trust_data, turn, ax, title):
    t_slice = trust_data[trust_data['turn'] == turn]
    if t_slice.empty:
        ax.set_title(f'{title} (no data)'); return
    pivot = t_slice.pivot_table(index='observer_id', columns='target_id', values='trust_value', aggfunc='first')
    sns.heatmap(pivot, annot=True, fmt='.0f', cmap='RdYlGn', vmin=0, vmax=100, ax=ax, cbar=False)
    ax.set_title(title)

if BASELINE_ID and t_base is not None:
    max_turn = int(trust_df['turn'].max())
    mid_turn = max_turn // 2
    fig, axes = plt.subplots(2, 3, figsize=(22, 14))
    for col_idx, t in enumerate([1, mid_turn, max_turn]):
        plot_trust_heatmap(t_base, t, axes[0][col_idx], f'Baseline T={t}')
        plot_trust_heatmap(trust_df, t, axes[1][col_idx], f'Target T={t}')
    plt.suptitle('Trust Matrix Evolution: Baseline (top) vs Target (bottom)', fontsize=14)
    plt.tight_layout(); plt.show()

### 2.3 Summary Statistics (Δ Baseline → Scenario)

In [ ]:
if BASELINE_ID and comp_global is not None:
    agg_cols = ['global_deception_avg', 'global_coherence_avg', 'global_satisfaction_avg',
                'territories_changed_hands', 'units_created', 'units_destroyed', 'global_trade_volume']
    base_agg = g_base[agg_cols].mean()
    tgt_agg = global_df[agg_cols].mean()
    delta = tgt_agg - base_agg
    summary = pd.DataFrame({'Baseline': base_agg, 'Target': tgt_agg, 'Δ': delta, 'Δ%': (delta / base_agg * 100).round(1)})
    display(summary.style.format('{:.3f}').background_gradient(subset=['Δ'], cmap='RdYlGn_r'))

---
# 🔀 Chapter 3 (RQ2): Counterfactual Analysis
**RQ2**: *What is the causal impact of a forced strategic decision on the geopolitical trajectory?*

Compares a base timeline against a forked timeline with an XAI injection.

### 3.1 Divergence Visualization

In [ ]:
# Configure the fork point (adjust after running a forked simulation)
FORK_TURN = None  # Set to the turn where forking occurred

if BASELINE_ID and comp_global is not None:
    fig, axes = plt.subplots(2, 2, figsize=(18, 12))
    metrics = [
        ('global_deception_avg', 'Deception'),
        ('global_coherence_avg', 'Coherence'),
        ('global_satisfaction_avg', 'Satisfaction'),
        ('global_trade_volume', 'Trade Volume'),
    ]
    for i, (col, title) in enumerate(metrics):
        ax = axes[i // 2][i % 2]
        sns.lineplot(data=comp_global, x='turn', y=col, hue='sim', ax=ax, linewidth=2)
        if FORK_TURN is not None:
            ax.axvline(FORK_TURN, color='black', linestyle='--', alpha=0.7, label=f'Fork @ T={FORK_TURN}')
            ax.legend()
        ax.set_title(f'RQ2: {title}')
    plt.suptitle('Counterfactual Divergence', fontsize=16, y=1.01)
    plt.tight_layout(); plt.show()
else:
    print('⚠️ Set BASELINE_ID and TARGET_ID to forked simulation pair.')

### 3.2 Prompt Audit: Envelope Comparison at Fork Point

In [ ]:
def load_envelopes(sim_id, turn):
    """Load raw envelope JSON for a specific turn."""
    rows = query_main(
        'SELECT nation_id, envelope_json FROM envelopes WHERE simulation_id = ? AND turn = ? ORDER BY nation_id',
        [sim_id, turn]
    )
    return rows

if BASELINE_ID and FORK_TURN is not None:
    # Compare envelopes at T+1 (first divergent turn)
    audit_turn = FORK_TURN + 1
    env_base = load_envelopes(BASELINE_ID, audit_turn)
    env_fork = load_envelopes(TARGET_ID, audit_turn)
    
    print(f'=== Envelope Audit at Turn {audit_turn} ===')
    for _, row in env_base.iterrows():
        nation = row['nation_id']
        fork_row = env_fork[env_fork['nation_id'] == nation]
        if fork_row.empty: continue
        
        base_env = json_lib.loads(row['envelope_json']) if isinstance(row['envelope_json'], str) else row['envelope_json']
        fork_env = json_lib.loads(fork_row.iloc[0]['envelope_json']) if isinstance(fork_row.iloc[0]['envelope_json'], str) else fork_row.iloc[0]['envelope_json']
        
        print(f'\n--- {nation} ---')
        # Compare public statements
        base_pub = base_env.get('public_statement', 'N/A')
        fork_pub = fork_env.get('public_statement', 'N/A')
        if base_pub != fork_pub:
            print(f'  PUBLIC STMT (base): {base_pub[:200]}')
            print(f'  PUBLIC STMT (fork): {fork_pub[:200]}')
        
        # Compare intents
        for domain in ['defense', 'foreign', 'economy']:
            bi = base_env.get(f'private_intent_{domain}', base_env.get(f'{domain}_private_intent', 'N/A'))
            fi = fork_env.get(f'private_intent_{domain}', fork_env.get(f'{domain}_private_intent', 'N/A'))
            if bi != fi:
                print(f'  {domain.upper()} intent: {bi} → {fi}')
else:
    print('⚠️ Set FORK_TURN to enable prompt audit.')

### 3.3 Cumulative Divergence Table

In [ ]:
if BASELINE_ID and comp_global is not None and FORK_TURN is not None:
    post_fork_base = g_base[g_base['turn'] > FORK_TURN]
    post_fork_tgt = global_df[global_df['turn'] > FORK_TURN]
    
    if not post_fork_base.empty and not post_fork_tgt.empty:
        merge = post_fork_base.merge(post_fork_tgt, on='turn', suffixes=('_base', '_fork'))
        divergence = {}
        for col in ['global_deception_avg', 'global_coherence_avg', 'global_satisfaction_avg', 'global_trade_volume']:
            diff = (merge[f'{col}_fork'] - merge[f'{col}_base']).abs()
            divergence[col] = {'Mean Δ': diff.mean(), 'Max Δ': diff.max(), 'Cumulative Σ|Δ|': diff.sum()}
        display(pd.DataFrame(divergence).T.style.format('{:.3f}'))
elif FORK_TURN is None:
    print('⚠️ Set FORK_TURN above.')

---
# 🛡️ Chapter 4 (RQ3): Strategic Deception & Moral Washing
**RQ3**: *Do LLM agents employ strategic deception and moral washing to justify predatory behavior?*

### 4.1 Deception by Domain (Defense vs Foreign)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

gap_df = nation_df.groupby('turn')[['deception_foreign', 'deception_defense']].mean().reset_index()
axes[0].plot(gap_df['turn'], gap_df['deception_defense'], label='Defense Domain', color=COLORS['defense'], linewidth=2)
axes[0].plot(gap_df['turn'], gap_df['deception_foreign'], label='Foreign Domain', color=COLORS['foreign'], linewidth=2)
axes[0].set_title('Avg Deception per Domain')
axes[0].set_ylabel('Deception Score'); axes[0].legend()

sns.lineplot(data=nation_df, x='turn', y='deception_overall', hue='nation_id', ax=axes[1], linewidth=1.5)
axes[1].set_title('Per-Nation Total Deception')
axes[1].set_ylim(-0.05, 1.05)
plt.tight_layout(); plt.show()

### 4.2 Deception Breakdown by GovernmentType

In [ ]:
if 'government_type' in behaviors_df.columns and behaviors_df['government_type'].notna().any():
    fig, axes = plt.subplots(1, 2, figsize=(18, 7))
    
    sns.boxplot(data=behaviors_df, x='government_type', y='deception_total', ax=axes[0], palette='Set2')
    axes[0].set_title('Deception by Government Type')
    axes[0].set_ylabel('Deception Score')
    
    # Time-series by gov type
    gov_ts = behaviors_df.groupby(['turn', 'government_type'])['deception_total'].mean().reset_index()
    sns.lineplot(data=gov_ts, x='turn', y='deception_total', hue='government_type', ax=axes[1], linewidth=2)
    axes[1].set_title('Deception Trend by Government Type')
    plt.tight_layout(); plt.show()
else:
    print('⚠️ government_type not available in behaviors table.')

### 4.3 Deception Breakdown by GlobalStrategy

In [ ]:
if 'global_strategy' in behaviors_df.columns and behaviors_df['global_strategy'].notna().any():
    fig, axes = plt.subplots(1, 2, figsize=(18, 7))
    
    sns.boxplot(data=behaviors_df, x='global_strategy', y='deception_total', ax=axes[0], palette='husl')
    axes[0].set_title('Deception by Global Strategy')
    axes[0].tick_params(axis='x', rotation=20)
    
    sns.boxplot(data=behaviors_df, x='global_strategy', y='coherence_score', ax=axes[1], palette='husl')
    axes[1].set_title('Coherence by Global Strategy')
    axes[1].tick_params(axis='x', rotation=20)
    plt.tight_layout(); plt.show()
else:
    print('⚠️ global_strategy not available in behaviors table.')

### 4.4 Deception × Coherence Scatter

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
hue_col = 'government_type' if 'government_type' in behaviors_df.columns and behaviors_df['government_type'].notna().any() else None
sns.scatterplot(data=behaviors_df, x='coherence_score', y='deception_total', hue=hue_col, alpha=0.5, ax=ax)
ax.set_xlabel('Coherence Score'); ax.set_ylabel('Deception Score')
ax.set_title('Deception vs Coherence (per nation-turn)')
ax.axhline(0.5, color='gray', linestyle='--', alpha=0.3)
ax.axvline(0.5, color='gray', linestyle='--', alpha=0.3)
plt.show()

### 4.5 Public Statement Text Mining
Keyword frequency analysis on public statements to detect moral washing patterns.

In [ ]:
# Extract public statements from envelopes
all_env = query_main(
    'SELECT e.nation_id, e.turn, e.envelope_json, b.government_type, b.global_strategy '
    'FROM envelopes e LEFT JOIN behaviors b ON e.simulation_id = b.simulation_id AND e.turn = b.turn AND e.nation_id = b.nation_id '
    'WHERE e.simulation_id = ?', [TARGET_ID]
)

statements = []
for _, row in all_env.iterrows():
    try:
        env = json_lib.loads(row['envelope_json']) if isinstance(row['envelope_json'], str) else row['envelope_json']
        stmt = env.get('public_statement', '')
        if stmt:
            statements.append({
                'nation_id': row['nation_id'], 'turn': row['turn'],
                'government_type': row.get('government_type', 'Unknown'),
                'statement': stmt
            })
    except: pass

stmt_df = pd.DataFrame(statements)
print(f'Extracted {len(stmt_df)} public statements.')

if not stmt_df.empty:
    # Moral washing keywords by governance type
    MORAL_KEYWORDS = {
        'liberation': ['liberat', 'freedom', 'free the', 'emancipat', 'democra'],
        'sacred': ['sacred', 'divine', 'holy', 'god', 'faith', 'righteous', 'mandate'],
        'security': ['security', 'stability', 'order', 'protect', 'defense', 'defend', 'deter'],
        'aggression': ['conquer', 'destroy', 'crush', 'dominate', 'annihilat', 'invade']
    }
    
    counts = {cat: [] for cat in MORAL_KEYWORDS}
    for gov_type in stmt_df['government_type'].unique():
        subset = stmt_df[stmt_df['government_type'] == gov_type]['statement'].str.lower().str.cat(sep=' ')
        for cat, kws in MORAL_KEYWORDS.items():
            count = sum(subset.count(kw) for kw in kws)
            counts[cat].append({'government_type': gov_type, 'count': count})
    
    # Plot keyword frequency by governance type
    kw_df = pd.DataFrame([item for sublist in counts.values() for item in sublist])
    kw_df['category'] = [cat for cat in MORAL_KEYWORDS for _ in range(len(stmt_df['government_type'].unique()))]
    
    fig, ax = plt.subplots(figsize=(14, 7))
    sns.barplot(data=kw_df, x='category', y='count', hue='government_type', ax=ax, palette='Set2')
    ax.set_title('Moral Washing: Keyword Frequency in Public Statements by Government Type')
    ax.set_ylabel('Keyword Occurrences'); ax.set_xlabel('Rhetorical Category')
    plt.show()

### 4.6 Public vs Private Intent Examples (High-Deception Turns)

In [ ]:
# Estrazione delle strategie ingannevoli uniche (rimozione duplicati consecutivi)
# Vogliamo mostrare un set variegato: massimo 2 esempi per singola Nazione
# e massimo 3 esempi per la stessa esatta "Firma" di intenti
high_deception = behaviors_df[behaviors_df['deception_total'] >= 0.6].sort_values('deception_total', ascending=False)

intent_examples = []
nation_counts = {}
signature_counts = {}

for _, row in high_deception.iterrows():
    if len(intent_examples) >= 10: break
    nation = row['nation_id']
    
    env_row = query_main(
        'SELECT envelope_json FROM envelopes WHERE simulation_id = ? AND turn = ? AND nation_id = ?',
        [TARGET_ID, int(row['turn']), nation]
    )
    if env_row.empty: continue
    
    try:
        env_raw = env_row.iloc[0]['envelope_json']
        env = json_lib.loads(env_raw) if isinstance(env_raw, str) else env_raw
        
        pub_def = str(env.get('defense_public_intent', env.get('public_intent_defense', '?')))
        priv_def = str(env.get('defense_private_intent', env.get('private_intent_defense', '?')))
        pub_for = str(env.get('foreign_public_intent', env.get('public_intent_foreign', '?')))
        priv_for = str(env.get('foreign_private_intent', env.get('private_intent_foreign', '?')))
        
        # Firma della strategia
        strategy_sig = (pub_def, priv_def, pub_for, priv_for)
        
        n_count = nation_counts.get(nation, 0)
        s_count = signature_counts.get(strategy_sig, 0)
        
        # Vogliamo un sample eterogeneo: max 3 per nazione, max 3 per stessa firma
        if n_count < 3 and s_count < 3:
            nation_counts[nation] = n_count + 1
            signature_counts[strategy_sig] = s_count + 1
            
            intent_examples.append({
                'Turn': int(row['turn']),
                'Nation': nation,
                'Gov': row.get('government_type', '?'),
                'Strat': row.get('global_strategy', '?'),
                'Deception': f"{row['deception_total']:.2f}",
                'Public Defense': pub_def[:40],
                'Private Defense': priv_def[:40],
                'Public Foreign': pub_for[:40],
                'Private Foreign': priv_for[:40],
            })
    except: pass

if intent_examples:
    display(pd.DataFrame(intent_examples))
else:
    print('No high-deception examples found.')


---
# 🌐 Chapter 5 (RQ4): Emergent Socio-Political Equilibria
**RQ4**: *Does the system converge toward stable cooperation or escalate toward chaotic conflict?*

### 5.1 Network Evolution Snapshots

In [ ]:
max_turn = int(trust_df['turn'].max()) if not trust_df.empty else 0
snapshot_turns = sorted(set([1, max_turn // 4, max_turn // 2, 3 * max_turn // 4, max_turn]))
# Use available snapshot turns
snapshot_turns = [t for t in snapshot_turns if t in trust_df['turn'].values]
if not snapshot_turns and not trust_df.empty:
    snapshot_turns = [trust_df['turn'].min(), trust_df['turn'].max()]

COLORS_REL = {'MUTUAL_DEFENSE': 'green', 'NON_AGGRESSION': '#FFD700', 'WAR': 'red', 'PEACE': 'lightgray'}
REL_PRIORITY = {'WAR': 3, 'MUTUAL_DEFENSE': 2, 'NON_AGGRESSION': 1, 'PEACE': 0}

def build_network(trust_data, turn):
    t_slice = trust_data[trust_data['turn'] == turn]
    G = nx.Graph()
    for _, row in t_slice.iterrows():
        u, v = row['observer_id'], row['target_id']
        if u == v: continue
        edge = tuple(sorted([u, v]))
        rel = row.get('relationship_state', 'PEACE')
        color = COLORS_REL.get(rel, 'gray')
        if G.has_edge(*edge):
            cur_rel = G[edge[0]][edge[1]].get('rel', 'PEACE')
            if REL_PRIORITY.get(rel, 0) > REL_PRIORITY.get(cur_rel, 0):
                G.add_edge(*edge, color=color, rel=rel)
        else:
            G.add_edge(*edge, color=color, rel=rel)
    return G

if not trust_df.empty:
    fig, axes = plt.subplots(1, len(snapshot_turns), figsize=(6 * max(1, len(snapshot_turns)), 6))
    if len(snapshot_turns) == 1: axes = [axes]
    
    # Use consistent layout mapping ALL nodes across the whole simulation time
    # This avoids KeyError when a nation dies and is removed, or a rebel nation spawns later
    G_all = nx.Graph()
    for _, row in trust_df.iterrows():
        u, v = row['observer_id'], row['target_id']
        if u != v:
            G_all.add_edge(u, v)
    
    # Generate positions for the comprehensive graph
    pos = nx.spring_layout(G_all, seed=42)

    for idx, t in enumerate(snapshot_turns):
        G = build_network(trust_df, t)
        if len(G.nodes) == 0:
            axes[idx].set_title(f'Turn {t} (No data)')
            axes[idx].axis('off')
            continue
            
        colors = [G[u][v].get('color', 'gray') for u, v in G.edges()]
        nx.draw(G, pos, with_labels=True, node_color='lightblue', edge_color=colors,
                ax=axes[idx], node_size=2000, width=2, font_size=8)
        axes[idx].set_title(f'Turn {t}')

    # Legend
    handles = [mlines.Line2D([], [], color=c, linewidth=2, label=l) for l, c in COLORS_REL.items()]
    fig.legend(handles=handles, loc='lower center', ncol=4, fontsize=10)
    plt.suptitle('Network Evolution', fontsize=14)
    plt.tight_layout(rect=[0, 0.06, 1, 0.96]); plt.show()
else:
    print('[INFO] No trust data available to plot networks.')


### 5.2 Alliance Bloc Detection

In [ ]:
# Build alliance-only subgraph at last turn
G_alliances = nx.Graph()
t_last = trust_df[trust_df['turn'] == max_turn]
for _, row in t_last.iterrows():
    if row.get('relationship_state') in ('MUTUAL_DEFENSE', 'NON_AGGRESSION'):
        G_alliances.add_edge(row['observer_id'], row['target_id'])

# Add isolated nations
all_nations = trust_df['observer_id'].unique()
for n in all_nations:
    if n not in G_alliances:
        G_alliances.add_node(n)

# Connected components = alliance blocs
blocs = list(nx.connected_components(G_alliances))
print(f'Detected {len(blocs)} alliance bloc(s) at Turn {max_turn}:')
for i, bloc in enumerate(blocs):
    print(f'  Bloc {i+1}: {sorted(bloc)}')

# Optionally try Louvain if networkx has it
try:
    from networkx.algorithms.community import louvain_communities
    communities = louvain_communities(G_alliances, seed=42)
    print(f'\nLouvain communities: {len(communities)}')
    for i, comm in enumerate(communities):
        print(f'  Community {i+1}: {sorted(comm)}')
except ImportError:
    pass

### 5.3 War Count & Violence Rate Over Time

In [ ]:
# Count active wars per turn from trust_df
wars_per_turn = trust_df[trust_df['relationship_state'] == 'WAR'].groupby('turn').size().reset_index(name='war_edges')
# Each war is counted twice (A->B and B->A), so divide by 2
wars_per_turn['active_wars'] = wars_per_turn['war_edges'] // 2

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

axes[0].bar(wars_per_turn['turn'], wars_per_turn['active_wars'], color='red', alpha=0.7)
axes[0].set_title('Active Wars per Turn')
axes[0].set_xlabel('Turn'); axes[0].set_ylabel('Number of Wars')

# Violence rate
vr = global_df[['turn', 'units_created', 'units_destroyed']].copy()
vr['violence_rate'] = vr['units_destroyed'] / vr['units_created'].replace(0, np.nan)
axes[1].plot(vr['turn'], vr['violence_rate'], color='darkred', linewidth=2)
axes[1].fill_between(vr['turn'], vr['violence_rate'], alpha=0.2, color='red')
axes[1].set_title('Violence Rate (Units Destroyed / Created)')
axes[1].set_xlabel('Turn'); axes[1].set_ylabel('Ratio')
axes[1].axhline(1.0, color='black', linestyle='--', alpha=0.3, label='Breakeven')
axes[1].legend()
plt.tight_layout(); plt.show()

### 5.4 Satisfaction Stability & Civil Unrest

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(22, 7))

# 5.4a Satisfaction per nation
sns.lineplot(data=nation_df, x='turn', y='public_satisfaction', hue='nation_id', ax=axes[0], linewidth=2)
axes[0].axhline(20, color='orange', linestyle='--', alpha=0.5, label='Strike threshold')
axes[0].axhline(10, color='red', linestyle='--', alpha=0.5, label='Unrest threshold')
axes[0].set_title('Satisfaction per Nation'); axes[0].legend(fontsize=8)

# 5.4b Global satisfaction variance (stability index)
stab = nation_df.groupby('turn')['public_satisfaction'].agg(['mean', 'std']).reset_index()
axes[1].plot(stab['turn'], stab['std'], color='purple', linewidth=2)
axes[1].fill_between(stab['turn'], stab['std'], alpha=0.2, color='purple')
axes[1].set_title('Satisfaction Dispersion (σ) — Lower = More Equal')
axes[1].set_xlabel('Turn'); axes[1].set_ylabel('Std Dev')

# 5.4c Civil unrest count per turn
if 'in_civil_unrest' in nation_df.columns:
    unrest = nation_df.groupby('turn')['in_civil_unrest'].sum().reset_index(name='nations_in_unrest')
    axes[2].bar(unrest['turn'], unrest['nations_in_unrest'], color='red', alpha=0.7)
    axes[2].set_title('Nations in Civil Unrest per Turn')
    axes[2].set_xlabel('Turn'); axes[2].set_ylabel('Count')
plt.tight_layout(); plt.show()

### 5.5 Territory Changes (Cumulative)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
cum_terr = global_df[['turn', 'territories_changed_hands']].copy()
cum_terr['cumulative'] = cum_terr['territories_changed_hands'].cumsum()
ax.fill_between(cum_terr['turn'], cum_terr['cumulative'], alpha=0.3, color='orange')
ax.plot(cum_terr['turn'], cum_terr['cumulative'], color='orange', linewidth=2)
ax.set_title('Cumulative Territory Changes')
ax.set_xlabel('Turn'); ax.set_ylabel('Provinces Changed Hands')
plt.show()

---
# 🔧 Chapter 6: Technical Monitoring
Token usage, costs, and infrastructure health.

In [ ]:
try:
    tokens = query_main(
        'SELECT turn, agent_type, SUM(prompt_tokens) as prompt, SUM(completion_tokens) as completion, '
        'SUM(total_tokens) as total, SUM(cost) as cost '
        'FROM token_usage WHERE simulation_id = ? GROUP BY turn, agent_type ORDER BY turn', [TARGET_ID]
    )
    
    if not tokens.empty:
        fig, axes = plt.subplots(1, 2, figsize=(18, 7))
        
        sns.lineplot(data=tokens, x='turn', y='total', hue='agent_type', ax=axes[0], linewidth=2)
        axes[0].set_title('Token Usage per Agent Type')
        
        cost_per_turn = tokens.groupby('turn')['cost'].sum().reset_index()
        cost_per_turn['cumulative_cost'] = cost_per_turn['cost'].cumsum()
        axes[1].plot(cost_per_turn['turn'], cost_per_turn['cumulative_cost'], linewidth=2, color='green')
        axes[1].set_title(f"Cumulative Cost (Total: ${cost_per_turn['cumulative_cost'].iloc[-1]:.4f})")
        axes[1].set_ylabel('USD')
        
        plt.tight_layout(); plt.show()
        
        # Summary table
        summary = tokens.groupby('agent_type').agg(
            total_tokens=('total', 'sum'),
            total_cost=('cost', 'sum'),
            avg_per_turn=('total', 'mean')
        ).sort_values('total_tokens', ascending=False)
        display(summary.style.format({'total_tokens': '{:,.0f}', 'total_cost': '${:.4f}', 'avg_per_turn': '{:,.0f}'}))
    else:
        print('No token usage data found.')
except Exception as e:
    print(f'Token usage query failed: {e}')

---
# ⚖️ Chapter 7: Action Governance Metrics
Engine-level execution outcomes and presidential approve/veto decisions.
Answers:
- Which action types succeed or fail most often, and why?
- Which domains does the President tend to veto?
- Are there asymmetries between nations in their governance style?

In [ ]:
# Load action outcomes and presidential decisions tables
def query_metrics(sql, params=None):
    """Query the metrics DuckDB (simulation_metrics.duckdb)."""
    conn = duckdb.connect(METRICS_DB)
    try:
        result = conn.execute(sql, params or []).df()
    except Exception as e:
        print(f'[METRICS QUERY ERROR] {e}')
        result = pd.DataFrame()
    finally:
        conn.close()
    return result

outcomes_df = query_metrics(
    'SELECT * FROM metrics_action_outcomes WHERE simulation_id = ?', [TARGET_ID]
)
decisions_df = query_metrics(
    'SELECT * FROM metrics_presidential_decisions WHERE simulation_id = ?', [TARGET_ID]
)

print(f'Action Outcomes rows : {len(outcomes_df)}')
print(f'Presidential Decisions rows: {len(decisions_df)}')
if not outcomes_df.empty:
    display(outcomes_df.head(5))
if not decisions_df.empty:
    display(decisions_df.head(5))

### 7.1 Action Outcome Rate by Domain & Action Type
How often does each action type succeed vs. fail at the engine level?

In [ ]:
if not outcomes_df.empty:
    summary = (
        outcomes_df
        .groupby(['domain', 'action_type', 'status'])
        .size()
        .reset_index(name='count')
    )

    # Pivot to wide form: columns = status values
    pivot = summary.pivot_table(
        index=['domain', 'action_type'],
        columns='status',
        values='count',
        fill_value=0
    ).reset_index()

    # Compute accept rate
    status_cols = [c for c in pivot.columns if c not in ('domain', 'action_type')]
    pivot['total'] = pivot[status_cols].sum(axis=1)
    if 'SUCCESS' in pivot.columns:
        pivot['success_rate_%'] = (pivot['SUCCESS'] / pivot['total'] * 100).round(1)

    print('=== Engine Acceptance Table ===')
    display(pivot.sort_values(['domain', 'action_type']))

    # Stacked bar chart
    fig, ax = plt.subplots(figsize=(16, 6))
    colors = {'SUCCESS': '#2ecc71', 'FAILED': '#e74c3c', 'PARTIAL': '#f39c12', 'PENDING': '#95a5a6'}
    bottom = None
    labels = [f"{r['domain']}/{r['action_type']}" for _, r in pivot.iterrows()]

    for status in [s for s in ['SUCCESS', 'FAILED', 'PARTIAL', 'PENDING'] if s in pivot.columns]:
        vals = pivot[status].values
        ax.bar(labels, vals, bottom=bottom, label=status,
               color=colors.get(status, '#bdc3c7'), edgecolor='white')
        bottom = vals if bottom is None else bottom + vals

    ax.set_title('Engine Action Outcomes — Stacked by Status', fontsize=14, fontweight='bold')
    ax.set_xlabel('Domain / Action Type')
    ax.set_ylabel('Count')
    ax.legend(loc='upper right')
    plt.xticks(rotation=35, ha='right')
    plt.tight_layout()
    plt.show()
else:
    print('[INFO] No action outcomes data for this simulation.')

### 7.2 Failure Reasons — Why Did Actions Fail?
Top failure reasons from the engine, grouped by domain.

In [ ]:
if not outcomes_df.empty:
    failed = outcomes_df[outcomes_df['status'] == 'FAILED'].copy()
    if not failed.empty:
        import re
        
        def bin_reason(text):
            if not isinstance(text, str): return str(text)
            
            # Province ownership
            if re.match(r'Province \d+ not owned by [A-Z]+', text):
                return 'Target province not owned'
            if re.match(r'Cannot deploy to non-owned province \d+', text):
                return 'Target province not owned'
                
            # Pathfinding
            if re.match(r'No valid path from \d+ to \d+ for [A-Z]+', text):
                matched = re.search(r'for ([A-Z]+)', text)
                unit = matched.group(1) if matched else 'UNIT'
                return f'No valid path for {unit}'
                
            # Distance Range
            if re.match(r'Distance \d+ exceeds [A-Z]+ range of \d+', text):
                matched = re.search(r'exceeds ([A-Z]+) range', text)
                unit = matched.group(1) if matched else 'UNIT'
                return f'Distance exceeds {unit} range'
                
            # Insufficient resources generic
            if re.match(r'Insufficient \w+[\.:] [Nn]eed \d+(\.\d+)?, [Hh]ave \d+(\.\d+)?', text) or \
               re.match(r'Insufficient \w+[\.:]? [Rr]equired: \d+(\.\d+)?, [Aa]vailable: \d+(\.\d+)?', text) or \
               re.match(r'Insufficient .*', text):
                res_match = re.search(r'Insufficient ([a-z]+)', text.lower())
                res = res_match.group(1) if res_match else 'resources'
                return f'Insufficient {res}'
                
            # Generic catch-all for missing IDs if any
            text = re.sub(r'province \d+', 'province', text, flags=re.IGNORECASE)
            
            return text

        failed['binned_reason'] = failed['reason'].apply(bin_reason)

        reason_counts = (
            failed
            .groupby(['domain', 'binned_reason'])
            .size()
            .reset_index(name='count')
            .sort_values('count', ascending=False)
        )

        # Top 15 reasons
        top_reasons = reason_counts.head(15)

        fig, ax = plt.subplots(figsize=(16, max(5, len(top_reasons) * 0.45)))
        colors_domain = {'Defense': '#e74c3c', 'Economy': '#3498db', 'Foreign': '#9b59b6'}
        bar_colors = [colors_domain.get(r, '#7f8c8d') for r in top_reasons['domain']]
        ax.barh(top_reasons['binned_reason'], top_reasons['count'], color=bar_colors, edgecolor='white')
        ax.set_xlabel('Failure Count')
        ax.set_title('Top Action Failure Reasons (Engine Level)', fontsize=14, fontweight='bold')
        ax.invert_yaxis()

        # Color legend
        from matplotlib.patches import Patch
        legend_handles = [Patch(facecolor=c, label=d) for d, c in colors_domain.items()]
        ax.legend(handles=legend_handles, loc='lower right')

        plt.tight_layout()
        plt.show()

        print('\n=== Failure Reasons Table ===')
        display(reason_counts)
    else:
        print('[INFO] No FAILED actions in this simulation.')
else:
    print('[INFO] No action outcomes data.')


### 7.3 Outcome Breakdown per Nation
Which nations have the highest failure rate? Are some nations more action-efficient?

In [ ]:
if not outcomes_df.empty:
    nation_summary = (
        outcomes_df
        .groupby(['nation_id', 'status'])
        .size()
        .reset_index(name='count')
        .pivot_table(index='nation_id', columns='status', values='count', fill_value=0)
        .reset_index()
    )
    status_cols = [c for c in nation_summary.columns if c not in ('nation_id',)]
    nation_summary['total'] = nation_summary[status_cols].sum(axis=1)
    if 'SUCCESS' in nation_summary.columns:
        nation_summary['success_rate_%'] = (
            nation_summary['SUCCESS'] / nation_summary['total'] * 100
        ).round(1)
    nation_summary = nation_summary.sort_values('success_rate_%', ascending=False)

    print('=== Per-Nation Engine Acceptance Rate ===')
    display(nation_summary)

    if 'success_rate_%' in nation_summary.columns:
        fig, ax = plt.subplots(figsize=(14, 5))
        colors_bar = ['#2ecc71' if r >= 70 else '#e67e22' if r >= 40 else '#e74c3c'
                      for r in nation_summary['success_rate_%']]
        ax.bar(nation_summary['nation_id'], nation_summary['success_rate_%'],
               color=colors_bar, edgecolor='white')
        ax.axhline(50, color='grey', linestyle='--', linewidth=1, label='50% threshold')
        ax.set_title('Engine Action Success Rate by Nation', fontsize=14, fontweight='bold')
        ax.set_ylabel('Success Rate (%)')
        ax.set_ylim(0, 105)
        plt.xticks(rotation=30, ha='right')
        plt.tight_layout()
        plt.show()
else:
    print('[INFO] No action outcomes data.')

### 7.4 Presidential Decision Rate (APPROVE vs VETO) by Domain
How controlling is each president? Which domain is most often vetoed?

In [ ]:
if not decisions_df.empty:
    # Overall APPROVE/VETO by domain
    domain_decisions = (
        decisions_df
        .groupby(['domain', 'decision'])
        .size()
        .reset_index(name='count')
        .pivot_table(index='domain', columns='decision', values='count', fill_value=0)
        .reset_index()
    )
    dec_cols = [c for c in domain_decisions.columns if c not in ('domain',)]
    domain_decisions['total'] = domain_decisions[dec_cols].sum(axis=1)
    if 'VETO' in domain_decisions.columns:
        domain_decisions['veto_rate_%'] = (
            domain_decisions['VETO'] / domain_decisions['total'] * 100
        ).round(1)

    print('=== Presidential Decision Summary by Domain ===')
    display(domain_decisions)

    # Grouped bar: APPROVE vs VETO per domain
    fig, axes = plt.subplots(1, 2, figsize=(18, 6))

    # Left: grouped bar APPROVE/VETO by domain
    domains = domain_decisions['domain'].values
    x = range(len(domains))
    width = 0.35
    approve_vals = domain_decisions.get('APPROVE', pd.Series([0]*len(domains))).values
    veto_vals = domain_decisions.get('VETO', pd.Series([0]*len(domains))).values
    axes[0].bar([i - width/2 for i in x], approve_vals, width, label='APPROVE', color='#2ecc71', edgecolor='white')
    axes[0].bar([i + width/2 for i in x], veto_vals, width, label='VETO', color='#e74c3c', edgecolor='white')
    axes[0].set_xticks(list(x))
    axes[0].set_xticklabels(domains)
    axes[0].set_title('Presidential Decisions by Domain', fontsize=13, fontweight='bold')
    axes[0].set_ylabel('Decision Count')
    axes[0].legend()

    # Right: Veto Rate by domain
    if 'veto_rate_%' in domain_decisions.columns:
        colors_veto = ['#e74c3c' if r > 30 else '#f39c12' if r > 10 else '#2ecc71'
                       for r in domain_decisions['veto_rate_%']]
        axes[1].bar(domains, domain_decisions['veto_rate_%'], color=colors_veto, edgecolor='white')
        axes[1].set_title('Veto Rate by Domain (%)', fontsize=13, fontweight='bold')
        axes[1].set_ylabel('Veto Rate (%)')
        axes[1].set_ylim(0, 105)
        axes[1].axhline(20, color='grey', linestyle='--', linewidth=1, label='20% reference')
        axes[1].legend()

    plt.tight_layout()
    plt.show()
else:
    print('[INFO] No presidential decisions data for this simulation.')

### 7.5 Presidential Veto Rate per Nation
Are some governments more interventionist than others?

In [ ]:
if not decisions_df.empty:
    nation_dec = (
        decisions_df
        .groupby(['nation_id', 'decision'])
        .size()
        .reset_index(name='count')
        .pivot_table(index='nation_id', columns='decision', values='count', fill_value=0)
        .reset_index()
    )
    dec_c = [c for c in nation_dec.columns if c not in ('nation_id',)]
    nation_dec['total'] = nation_dec[dec_c].sum(axis=1)
    if 'VETO' in nation_dec.columns:
        nation_dec['veto_rate_%'] = (nation_dec['VETO'] / nation_dec['total'] * 100).round(1)
    nation_dec = nation_dec.sort_values('veto_rate_%', ascending=False)

    print('=== Per-Nation Presidential Veto Rate ===')
    display(nation_dec)

    fig, ax = plt.subplots(figsize=(14, 5))
    veto_colors = ['#e74c3c' if r > 30 else '#f39c12' if r > 10 else '#2ecc71'
                   for r in nation_dec.get('veto_rate_%', pd.Series([]))]
    ax.bar(nation_dec['nation_id'], nation_dec.get('veto_rate_%', 0),
           color=veto_colors, edgecolor='white')
    ax.set_title('Presidential Veto Rate by Nation', fontsize=14, fontweight='bold')
    ax.set_ylabel('Veto Rate (%)')
    ax.set_ylim(0, 105)
    ax.axhline(20, color='grey', linestyle='--', linewidth=1, label='20% reference')
    ax.legend()
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.show()
else:
    print('[INFO] No presidential decisions data.')

### 7.6 Engine Outcome vs Presidential Decision — Are Vetoed Actions More Likely to Fail?
Cross-table: do APPROVEd actions have a higher engine success rate than if they'd been approved despite poor conditions?

In [ ]:
if not outcomes_df.empty and not decisions_df.empty:
    # Join on (simulation_id, turn, nation_id, domain)
    merged = outcomes_df.merge(
        decisions_df[['turn', 'nation_id', 'domain', 'decision']],
        on=['turn', 'nation_id', 'domain'],
        how='left'
    )

    if not merged.empty and 'decision' in merged.columns:
        cross = (
            merged
            .groupby(['decision', 'status'])
            .size()
            .reset_index(name='count')
            .pivot_table(index='decision', columns='status', values='count', fill_value=0)
            .reset_index()
        )
        s_cols = [c for c in cross.columns if c != 'decision']
        cross['total'] = cross[s_cols].sum(axis=1)
        if 'SUCCESS' in cross.columns:
            cross['success_rate_%'] = (cross['SUCCESS'] / cross['total'] * 100).round(1)

        print('=== Presidential Decision vs Engine Outcome ===')
        display(cross)
    else:
        print('[INFO] Merge produced no matching rows — tables may have data from different simulation runs.')
else:
    print('[INFO] Missing data for cross-analysis.')